In [1]:
%pip install gliner bert_score sentence_transformers osmnx


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import json
import pickle
import numpy as np
import torch
import networkx as nx
import osmnx as ox
from typing import Dict, List
from sentence_transformers import SentenceTransformer, util
from gliner import GLiNER
from tqdm import tqdm

# ==========================================
# 1. Independent Evaluation Suite (Critic)
# ==========================================
class prompt_based_route_evaluator:
    def __init__(self, graph, prompts: List[str], routes: List[List[int]], device: str = "cpu"):
        self.graph = graph
        self.prompts = prompts
        self.routes = routes
        # SWAP: Using a larger, QA-tuned model for independent verification
        self.st_evaluator = SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1', device=device)

    def check_path_validity(self, route: List[int]) -> bool:
        if not route or len(route) < 2: return False
        return all(self.graph.has_edge(u, v) for u, v in zip(route[:-1], route[1:]))

    def deviation_penalty(self, route: List[int]) -> float:
        if not self.check_path_validity(route): return 10.0
        start, end = route[0], route[-1]
        try:
            ideal_len = nx.shortest_path_length(self.graph, start, end, weight='length')
            actual_len = sum(self.graph[u][v][0].get('length', 1.0) for u, v in zip(route[:-1], route[1:]))
            return actual_len / ideal_len if ideal_len > 0 else 1.0
        except: return 10.0

    def constraint_satisfaction(self, prompt: str, route: List[int]) -> float:
        words = [w.lower() for w in prompt.split() if len(w) > 3]
        if not words: return 1.0
        
        path_tags = []
        for u, v in zip(route[:-1], route[1:]):
            data = self.graph.get_edge_data(u, v)[0]
            for key in ['name', 'highway', 'amenity', 'shop', 'leisure', 'landuse']:
                val = data.get(key)
                if val: path_tags.extend(val if isinstance(val, list) else [str(val)])
        
        if not path_tags: return 0.0
        
        c_embs = self.st_evaluator.encode(words, convert_to_tensor=True)
        t_embs = self.st_evaluator.encode(list(set(path_tags)), convert_to_tensor=True)
        scores = util.cos_sim(c_embs, t_embs)
        return torch.sum(torch.max(scores, dim=1)[0] > 0.60).item() / len(words)

    def evaluate_method(self) -> Dict:
        return {
            "avg_path_validity": np.mean([self.check_path_validity(r) for r in self.routes]),
            "avg_deviation_penalty": np.mean([self.deviation_penalty(r) for r in self.routes]),
            "avg_constraint_satisfaction": np.mean([self.constraint_satisfaction(p, r) for p, r in zip(self.prompts, self.routes)])
        }

# ==========================================
# 2. Semantic Fuzzy-Router (Actor)
# ==========================================
class GlinerAStarRouter:
    def __init__(self, graph, model_name="urchade/gliner_medium-v2.1"):
        self.graph = graph
        self.gliner_model = GLiNER.from_pretrained(model_name)
        # Actor Model: all-MiniLM-L6-v2
        self.st_router = SentenceTransformer('all-MiniLM-L6-v2')
        self.labels = ["road_type_to_avoid", "amenity_required"]
        
        # Pre-cache unique tags to prevent model-looping during Dijkstra
        print("Enriching graph semantic index...")
        unique_tags = set()
        for _, _, data in graph.edges(data=True):
            for k in ['name', 'highway', 'amenity', 'shop', 'leisure', 'landuse']:
                v = data.get(k)
                if v and str(v) != 'nan':
                    unique_tags.add(str(v).lower())
        
        self.tag_list = list(unique_tags)
        self.tag_embeddings = self.st_router.encode(self.tag_list, convert_to_tensor=True)

    def find_route(self, start_node, end_node, prompt: str):
        entities = self.gliner_model.predict_entities(prompt, self.labels, threshold=0.55)
        if not entities:
            return nx.dijkstra_path(self.graph, start_node, end_node, weight='length')

        # Map GLiNER entities to OSM tags via 0.85 similarity threshold
        entity_texts = [ent['text'].lower() for ent in entities]
        entity_embs = self.st_router.encode(entity_texts, convert_to_tensor=True)
        cos_sim = util.cos_sim(entity_embs, self.tag_embeddings)
        
        trigger_map = {}
        for i, ent in enumerate(entities):
            # 0.85 Threshold prevents 'semantic noise' (verified by your 0.9 trial)
            match_indices = torch.where(cos_sim[i] > 0.85)[0]
            trigger_map[ent['text'].lower()] = {
                "tags": [self.tag_list[idx] for idx in match_indices],
                "label": ent['label']
            }

        def weight_func(u, v, edge_dict):
            data = list(edge_dict.values())[0] if isinstance(edge_dict, dict) else edge_dict[0]
            base_cost = data.get('length', 1.0)
            
            # Aggregate all metadata for the road segment
            edge_meta = " ".join([str(data.get(k, '')).lower() for k in ['name', 'highway', 'amenity', 'shop', 'leisure']])
            
            mult = 1.0
            for ent_text, info in trigger_map.items():
                # Fuzzy Check: Does the edge have a tag semantically linked to the entity?
                if any(t in edge_meta for t in info['tags']) or ent_text in edge_meta:
                    if info['label'] == "road_type_to_avoid":
                        mult *= 10000.0 # Aggressive penalty
                    elif info['label'] == "amenity_required":
                        mult *= 0.000001 # Aggressive reward
            return base_cost * mult

        try:
            return nx.dijkstra_path(self.graph, start_node, end_node, weight=weight_func)
        except:
            return None

# ==========================================
# 3. Execution Pipeline
# ==========================================
if __name__ == "__main__":
    graph_path = "research/pioneer_valley_v2.pkl"
    with open(graph_path, "rb") as f:
        G = pickle.load(f)

    with open("research/synthetic_dataset.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

    router = GlinerAStarRouter(G)
    gen_routes, gen_prompts = [], []

    print("\n--- Running Multi-Model Experiment ---")
    for item in tqdm(data[:30]):
        try:
            s_pt = ox.geocoder.geocode(f"{item['start']}, MA, USA")
            e_pt = ox.geocoder.geocode(f"{item['end']}, MA, USA")
            s_node = ox.distance.nearest_nodes(G, X=s_pt[1], Y=s_pt[0])
            e_node = ox.distance.nearest_nodes(G, X=e_pt[1], Y=e_pt[0])
            
            if s_node == e_node: continue

            route = router.find_route(s_node, e_node, item['prompt'])
            if route:
                gen_routes.append(route)
                gen_prompts.append(item['prompt'])
        except: continue

    if gen_routes:
        evaluator = prompt_based_route_evaluator(G, gen_prompts, gen_routes)
        print("\nFinal Results (Evaluated by MPNet):", evaluator.evaluate_method())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 245.82it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enriching graph semantic index...

--- Running Multi-Model Experiment ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1439.47it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Final Results (Evaluated by MPNet): {'avg_path_validity': 1.0, 'avg_deviation_penalty': 1.0011296507899876, 'avg_constraint_satisfaction': 0.17348917748917747}
